# 05 — Inventory-aware market making

**Project:** Avellaneda–Stoikov-lite quote simulator on a synthetic mid path.

## Problem
A market maker earns the spread but accumulates inventory risk when the mid
trends. How do inventory skew and fill intensity interact with PnL?

## Method
1. Simulate a mid path (`simulate_mid_path` GBM or OU).
2. Run `MarketMakerSimulator` with half-spread, inventory skew, and fill probs.
3. Decompose mark-to-market into spread vs inventory components.

## Modules
- `quant_lab.microstructure.market_maker`
- `quant_lab.data.sample`


In [ ]:
from quant_lab.microstructure import (
    MarketMakerConfig,
    MarketMakerSimulator,
    simulate_mid_path,
)

mid = simulate_mid_path(n_steps=400, x0=100.0, sigma=0.25, process="gbm", seed=3)
cfg = MarketMakerConfig(
    half_spread=0.05,
    inventory_skew=0.02,
    max_inventory=15,
    fill_prob=0.12,
    touch_boost=0.30,
    inventory_penalty=0.001,
    seed=3,
)
sim = MarketMakerSimulator(cfg)
result = sim.run(mid)
print(result.summary())
print(result.path.tail(3))


In [ ]:
from dataclasses import replace

flat = MarketMakerSimulator(
    replace(cfg, inventory_skew=0.0)
).run(mid)
print("with skew:", result.summary())
print("flat quotes:", flat.summary())


In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless-friendly
import matplotlib.pyplot as plt

path = result.path
fig, axes = plt.subplots(3, 1, figsize=(9, 6), sharex=True)
axes[0].plot(path["mid"], label="mid", lw=1)
axes[0].plot(path["bid"], label="bid", lw=0.8, alpha=0.8)
axes[0].plot(path["ask"], label="ask", lw=0.8, alpha=0.8)
axes[0].legend(loc="upper left")
axes[0].set_title("Quotes vs mid")
path["inventory"].plot(ax=axes[1], title="Inventory")
path["mtm_pnl"].plot(ax=axes[2], label="MTM")
path["spread_pnl"].plot(ax=axes[2], label="Spread cum")
axes[2].legend()
axes[2].set_title("PnL decomposition")
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Short result
Inventory skew typically reduces terminal inventory and inventory PnL volatility
versus flat quotes on the same mid path, sometimes at the cost of lower earned
spread. Exact numbers are seed-dependent — compare the two `summary()` lines.

## Limitations
- Bernoulli fill model is a toy; real queues, adverse selection, and latency dominate.
- Single-asset, discrete lots, no fees or exchange incentives.
- GBM mid ignores microstructure noise and discrete tick rules.
- Research/education only — not a live quoting engine.
